# Clase 9 — Casos de uso reales y cierre del curso

Curso **Deep Learning con Python y PyTorch desde cero** — DIH DATAlife

Duración estimada: 90 min

## Objetivos de aprendizaje

- Relacionar cada técnica del curso con aplicaciones reales (visión por computador, procesamiento de lenguaje natural, cómputo a gran escala en GPU, evaluación de modelos en producción)
- Repasar el patrón completo del curso (tensores → dataset → modelo → entrenamiento → evaluación) sobre un ejemplo integrador breve
- Diseñar la propuesta de un proyecto final propio combinando las técnicas vistas en el curso

Esta es la última clase de teoría del curso. No vas a aprender ninguna sintaxis nueva de PyTorch: vas a conectar todo lo que ya sabes con dominios de aplicación reales y vas a repasar, en una única celda de código, el recorrido completo que has hecho desde la clase 1.

## De ocho clases sueltas a un patrón único

Repasa el camino: en la clase 1 construiste tu primer tensor y tu primera red mínima. En la clase 2 aprendiste a representar datos con `Dataset` y a servirlos en lotes (batches) con `DataLoader`. En las clases 3 y 4 construiste una red densa con `nn.Module` y la entrenaste sobre MNIST (Modified National Institute of Standards and Technology, un banco de imágenes de dígitos escritos a mano). En las clases 5 y 6 especializaste esa misma arquitectura en una red neuronal convolucional (CNN, Convolutional Neural Network) para imágenes en color, sobre CIFAR-10. En la clase 7 aprendiste a acelerar ese entrenamiento con una GPU (Graphics Processing Unit). Y en la clase 8 aprendiste a mirar un modelo entrenado con ojo crítico: sobreajuste, matriz de confusión, guardado y carga de pesos.

Ese recorrido no son ocho técnicas sueltas: es un único patrón que se repite siempre igual, sea el problema que sea:

**tensores → dataset → modelo → entrenamiento → evaluación**

Lo único que cambia de un proyecto real a otro es el contenido de cada paso: qué datos entran, qué arquitectura tiene el modelo y qué métrica define el éxito. La sintaxis de PyTorch que usas para escribirlo es siempre la misma.

## Dominios de aplicación real, según el material de la certificación

El material oficial de la certificación EITC/AI/DLPP (European IT Certificate — Certificado Europeo de Tecnologías de la Información — en Inteligencia Artificial, especialidad Deep Learning with Python and PyTorch) menciona de forma constante los mismos dominios de aplicación a lo largo de sus ocho temas. La tabla siguiente conecta cada dominio con la clase del curso donde ya trabajaste la técnica que lo sostiene.

| Dominio | Qué usa de PyTorch | Clase del curso |
|---|---|---|
| Visión por computador | `torchvision`, redes convolucionales (CNN), tensores con canales RGB (Red-Green-Blue) | Clases 2, 5 y 6 |
| Procesamiento de lenguaje natural (NLP, Natural Language Processing) | `torchtext`, modelos transformer a gran escala como GPT-3 (Generative Pre-trained Transformer 3, con 175.000 millones de parámetros), integración con Hugging Face Transformers | Clases 3 y 4 (mismo patrón de modelo y entrenamiento, a otra escala) |
| Cómputo a gran escala en GPU | CUDA (Compute Unified Device Architecture), `torch.device`, GPU en la nube, computación de altas prestaciones (HPC, High-Performance Computing) | Clase 7 |
| Evaluación e interpretabilidad de modelos | Mapas de relevancia (saliency maps), análisis de atención, estimación de incertidumbre, matriz de confusión | Clase 8 |
| Despliegue en producción | `torch.quantization`, TorchScript, ONNX (Open Neural Network Exchange), TorchServe | Clase 8 (a partir de `torch.save`) |

Lee el capítulo 9 del manual si quieres el detalle completo de cada dominio, con las referencias exactas al material de la certificación.

## Lo que tienen en común

- Todos parten del mismo patrón: tensores, dataset, modelo, entrenamiento, evaluación, sin excepción
- La técnica se elige por el tipo de dato de entrada (imagen, texto, tabla de números), no al revés
- La escala cambia mucho más que la sintaxis: entrenar una red de tres capas sobre MNIST y entrenar un modelo de 175.000 millones de parámetros usan el mismo `nn.Module`, el mismo bucle de `forward`, `backward` y `step`, solo que el segundo necesita miles de GPU en paralelo
- Evaluar un modelo no termina en la precisión: en dominios críticos (conducción autónoma, salud, diagnóstico) hace falta interpretar por qué el modelo decide lo que decide, no solo cuánto acierta
- Un modelo entrenado no es el final: guardarlo con `torch.save` es el primer paso hacia herramientas de despliegue como TorchScript, ONNX o TorchServe

## Repaso integrador: el patrón completo en una celda

La celda de código siguiente reconstruye, en una única pasada, el patrón completo del curso sobre un ejemplo pequeño e intencionadamente breve: una red densa como la de las clases 3 y 4, entrenada durante un número reducido de pasos sobre MNIST, evaluada sobre un par de ejemplos que no ha visto durante el entrenamiento.

No es un modelo con buen rendimiento —entrena solo unos segundos, no busca precisión— es un resumen en código de las cinco fases del curso, para que las veas todas juntas una última vez antes del proyecto final:

1. **Tensores**: los datos de entrada y las etiquetas viajan siempre como tensores de PyTorch
2. **Dataset**: `torchvision.datasets.MNIST` te da acceso al dataset ya troceado en ejemplos individuales
3. **Modelo**: una clase que hereda de `nn.Module`, igual que en la clase 3
4. **Entrenamiento**: el bucle de `forward`, cálculo de la pérdida (loss), `backward` y `optimizador.step()` de la clase 4
5. **Evaluación**: `torch.no_grad()` y `torch.argmax` para predecir sobre ejemplos nuevos, igual que en la clase 8

En Google Colab, esta celda descarga MNIST la primera vez que la ejecutas (necesita conexión a internet). Si ya la habías descargado en una clase anterior, PyTorch reutiliza la copia local.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. TENSORES: transformamos cada imagen a un tensor antes de usarla
transformaciones = transforms.Compose([transforms.ToTensor()])

# 2. DATASET: reutilizamos MNIST, igual que en las clases 2, 3, 4 y 7
dataset_entrenamiento = datasets.MNIST(
    root="./datos", train=True, download=True, transform=transformaciones
)
dataset_test = datasets.MNIST(
    root="./datos", train=False, download=True, transform=transformaciones
)

# Nos quedamos con un subconjunto pequeño para que el repaso sea rápido:
# el objetivo de esta celda es repasar el patrón, no maximizar la precisión
subconjunto_entrenamiento = torch.utils.data.Subset(dataset_entrenamiento, range(2000))
cargador_entrenamiento = DataLoader(subconjunto_entrenamiento, batch_size=128, shuffle=True)

print("Ejemplos de entrenamiento usados en este repaso:", len(subconjunto_entrenamiento))
print("Ejemplos de test disponibles:", len(dataset_test))


# 3. MODELO: la misma red densa de tres capas de las clases 3, 4 y 7
class RedMNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa1 = nn.Linear(28 * 28, 128)
        self.capa2 = nn.Linear(128, 64)
        self.capa3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)      # aplana cada imagen 28x28 en un vector de 784 valores
        x = F.relu(self.capa1(x))
        x = F.relu(self.capa2(x))
        return self.capa3(x)           # logits en bruto: CrossEntropyLoss se encarga del resto


modelo = RedMNIST()
funcion_perdida = nn.CrossEntropyLoss()
optimizador = optim.Adam(modelo.parameters(), lr=0.001)

# 4. ENTRENAMIENTO: el mismo bucle de la clase 4, aquí solo dos épocas
modelo.train()
for epoca in range(2):
    perdida_epoca = 0.0
    for imagenes, etiquetas in cargador_entrenamiento:
        optimizador.zero_grad()
        salida = modelo(imagenes)
        perdida = funcion_perdida(salida, etiquetas)
        perdida.backward()
        optimizador.step()
        perdida_epoca += perdida.item()
    print(f"Época {epoca + 1}: pérdida media = {perdida_epoca / len(cargador_entrenamiento):.3f}")

# 5. EVALUACIÓN: predecimos sobre dos ejemplos de test que el modelo no ha visto
modelo.eval()
with torch.no_grad():
    for indice in [0, 1]:
        imagen, etiqueta_real = dataset_test[indice]
        salida = modelo(imagen.unsqueeze(0))   # unsqueeze añade la dimensión de lote (batch)
        prediccion = torch.argmax(salida, dim=1).item()
        print(f"Ejemplo {indice}: etiqueta real = {etiqueta_real}, predicción del modelo = {prediccion}")

Con solo dos épocas sobre 2.000 ejemplos no esperes una precisión alta: lo importante es que has visto, en una sola celda, las cinco fases del patrón completo del curso funcionando juntas. Si quieres una precisión seria, vuelve a las clases 4 y 6, donde entrenaste con el dataset completo durante más épocas.

Ese mismo patrón —con el modelo, el dataset y la escala que decidas tú— es exactamente lo que vas a construir en tu proyecto final.

## Y ahora, tu proyecto final

1. **Elige un problema y un dataset**: reutiliza MNIST o CIFAR-10, o busca uno propio en `torchvision.datasets` o en datos que generes tú mismo
2. **Elige la técnica según el tipo de dato**: imágenes → CNN (clases 5 y 6); datos tabulares → red densa (clases 3 y 4); si el volumen o el modelo crecen, entrena en GPU (clase 7)
3. **Estructura el entregable**: un notebook con el pipeline completo (carga, modelo, entrenamiento, evaluación con matriz de confusión, guardado con `torch.save`) más una memoria breve de una o dos páginas
4. **Presenta y defiende tu proyecto** en cinco a diez minutos: problema, dataset, arquitectura, resultados y limitaciones

Consulta el capítulo 9 del manual para leer el detalle completo de los dominios de aplicación y la guía paso a paso del proyecto final. Ahora abre el notebook de práctica de esta clase, `clase09_practica_casos_de_uso_reales.ipynb`: ahí encuentras el andamiaje inicial para arrancar tu propio proyecto.